# Independent Random Undersampling Experiment

**Purpose.** Test whether keeping all positive household high-demand events while randomly reducing negative training examples improves future-event ranking. P95 is the primary target; P90 is a secondary replication.

**Research boundary.** This is a separate development-only sensitivity experiment. It reads frozen project datasets and model settings, excludes intentional household holdouts from every development stage, never loads outer-test target values, and writes generated files only under this experiment's `outputs/` directory. It does not change the completed model-selection or dissertation evaluation results.

**Comparison.** Logistic Regression, Random Forest, XGBoost, and CatBoost use the same chronological folds and identical sampled row IDs for a given target, fold, ratio, and sampling seed. Validation data always retains its original prevalence. PR-AUC (average precision, matching Notebook 05) is the primary metric.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import warnings

import catboost
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import display
from sklearn.calibration import calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, brier_score_loss,
    f1_score, log_loss, precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "processed").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "data" / "processed").is_dir():
    raise RuntimeError("Open/run this notebook from the project or its notebooks folder.")

DATA_DIR = ROOT / "data" / "processed"
TRAINING_TABLE_DIR = ROOT / "reports" / "tables" / "05_model_training_tuning"
SPLIT_TABLE_DIR = ROOT / "reports" / "tables" / "04_split_design"
HOLDOUT_TABLE_DIR = ROOT / "reports" / "tables" / "04b_unseen_household_holdout_design"
EXP_DIR = ROOT / "notebooks" / "experiments" / "class_imbalance"
OUTPUT_DIR = EXP_DIR / "outputs"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"
MANIFEST_DIR = OUTPUT_DIR / "manifests"
for folder in (TABLE_DIR, FIGURE_DIR, MODEL_DIR, MANIFEST_DIR):
    folder.mkdir(parents=True, exist_ok=True)

TARGETS = ("P95", "P90")
FAMILIES = ("Logistic Regression", "Random Forest", "XGBoost", "CatBoost")
PRIMARY_RATIOS = {"1:5": 5, "1:4": 4, "1:3": 3}
SAMPLING_SEEDS = (42, 123, 2026)
MODEL_RANDOM_STATE = 42
N_JOBS = 2
THRESHOLD_GRID = np.round(np.arange(0.01, 1.00, 0.01), 2)

print("Project:", ROOT)
print("Experiment outputs:", OUTPUT_DIR)
print("Targets:", TARGETS, "| primary P95")
print("Primary undersampling ratios:", tuple(PRIMARY_RATIOS))
print("Sampling seeds:", SAMPLING_SEEDS, "| model random state:", MODEL_RANDOM_STATE)
print("Outer-test labels are not loaded or scored by this notebook.")

## Stage 1 — Frozen inputs and provenance

Read the exact model-ready rows and configurations used by Notebook 05. Contracts and manifests are checked before modeling. The row-ID scan selects eligible development rows before target/features are parsed, so outer-test labels stay out of memory.

In [ ]:
P95_PATH = DATA_DIR / "05_model_ready_m0_p95.csv"
P90_PATH = DATA_DIR / "05_model_ready_m0_p90.csv"
MASTER_PATH = DATA_DIR / "04_master_split.csv"
MASTER_CONTRACT_PATH = DATA_DIR / "04_master_split_dtype_contract.json"
HOLDOUT_PATH = DATA_DIR / "04b_household_holdout_assignment.csv"
HOLDOUT_CONTRACT_PATH = DATA_DIR / "04b_household_holdout_assignment_dtype_contract.json"
P95_CONTRACT_PATH = DATA_DIR / "05_model_ready_m0_p95_dtype_contract.json"
P90_CONTRACT_PATH = DATA_DIR / "05_model_ready_m0_p90_dtype_contract.json"
SPLIT_MANIFEST_PATH = SPLIT_TABLE_DIR / "04_split_design_manifest.json"
HOLDOUT_MANIFEST_PATH = HOLDOUT_TABLE_DIR / "04b_unseen_household_holdout_manifest.json"
FOLD_PATH = TRAINING_TABLE_DIR / "05_02_time_cv_fold_design.csv"
FAMILY_CONFIG_PATH = TRAINING_TABLE_DIR / "05_06_best_config_per_family.csv"
SELECTED_MODELS_PATH = TRAINING_TABLE_DIR / "05_selected_models.json"
TARGET_POLICY_PATH = ROOT / "reports" / "tables" / "03a_target_design_experiments" / "03a_selected_target_config.json"
FEATURE_POLICY_PATH = ROOT / "reports" / "tables" / "03b_historical_feature_quality_experiments" / "03b_selected_feature_quality_config.json"

REQUIRED_INPUTS = [P95_PATH, P90_PATH, MASTER_PATH, MASTER_CONTRACT_PATH,
    HOLDOUT_PATH, HOLDOUT_CONTRACT_PATH, P95_CONTRACT_PATH, P90_CONTRACT_PATH,
    SPLIT_MANIFEST_PATH, HOLDOUT_MANIFEST_PATH, FOLD_PATH, FAMILY_CONFIG_PATH,
    SELECTED_MODELS_PATH, TARGET_POLICY_PATH, FEATURE_POLICY_PATH]
missing_inputs = [str(p) for p in REQUIRED_INPUTS if not p.is_file()]
if missing_inputs:
    raise FileNotFoundError("Required frozen inputs are missing:\n" + "\n".join(missing_inputs))

def sha256(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

source_hashes = {str(p.relative_to(ROOT)): sha256(p) for p in REQUIRED_INPUTS}
print(f"Verified that all {len(REQUIRED_INPUTS)} frozen input files exist.")

In [ ]:
split_manifest = read_json(SPLIT_MANIFEST_PATH)
holdout_manifest = read_json(HOLDOUT_MANIFEST_PATH)
selected_models = read_json(SELECTED_MODELS_PATH)
target_policy = read_json(TARGET_POLICY_PATH)
feature_policy = read_json(FEATURE_POLICY_PATH)

if sha256(MASTER_PATH) != split_manifest["master_split"]["sha256"]:
    raise RuntimeError("Master split hash differs from frozen Notebook 04 manifest.")
for target, path in (("P95", P95_PATH), ("P90", P90_PATH)):
    key = f"{target.lower()}_model_dataset"
    sig = split_manifest["input_signatures"][key]
    if sha256(path) != sig["sha256"] or path.stat().st_size != sig["size_bytes"]:
        raise RuntimeError(f"{target} model-ready input differs from the frozen Notebook 04 signature.")
if sha256(HOLDOUT_PATH) != holdout_manifest["assignment"]["sha256"]:
    raise RuntimeError("Household assignment differs from frozen Notebook 04B manifest.")
if holdout_manifest.get("future_target_outcomes_used_for_assignment", True):
    raise RuntimeError("Household assignment provenance does not confirm outcome-independent assignment.")
if not holdout_manifest.get("validation", {}).get("assignment_groups_are_disjoint_and_complete", False):
    raise RuntimeError("Frozen household assignment did not pass Notebook 04B validation.")
if set(selected_models) != set(TARGETS):
    raise RuntimeError("Notebook 05 selected-model handoff must contain P95 and P90.")
if target_policy.get("status") != "frozen_primary_research_policy":
    raise RuntimeError("Frozen target policy is missing or has an unexpected status.")
if feature_policy.get("status") != "frozen_feature_quality_policy":
    raise RuntimeError("Frozen feature-quality policy is missing or has an unexpected status.")

family_configs = pd.read_csv(FAMILY_CONFIG_PATH, dtype={"target": "string", "model_family": "string", "parameters_json": "string"})
if set(family_configs.target.astype(str)) != set(TARGETS) or set(family_configs.model_family.astype(str)) != set(FAMILIES):
    raise RuntimeError("Per-family Notebook 05 configuration table is incomplete.")
family_params = {}
for row in family_configs.itertuples(index=False):
    key = (str(row.target), str(row.model_family))
    if key in family_params:
        raise RuntimeError(f"Duplicate frozen family configuration for {key}.")
    family_params[key] = json.loads(row.parameters_json)
if len(family_params) != len(TARGETS) * len(FAMILIES):
    raise RuntimeError("Expected one frozen best configuration per target and model family.")

master_contract = read_json(MASTER_CONTRACT_PATH)
assignment_contract = read_json(HOLDOUT_CONTRACT_PATH)
for path, contract in ((MASTER_PATH, master_contract), (HOLDOUT_PATH, assignment_contract)):
    header = pd.read_csv(path, nrows=0, encoding=contract["encoding"], sep=contract["separator"]).columns.tolist()
    if header != list(contract["columns"]):
        raise RuntimeError(f"CSV header differs from frozen dtype contract: {path.name}")
master = pd.read_csv(MASTER_PATH, dtype={"row_id": "string", "household_id": "string"})
assignment = pd.read_csv(HOLDOUT_PATH, dtype={"household_id": "string", "holdout_group": "string"})
master["target_date"] = pd.to_datetime(master["target_date"], errors="raise").dt.normalize()
if master.row_id.isna().any() or master.row_id.duplicated().any():
    raise RuntimeError("Frozen master split row_id must be unique and non-null.")
if assignment.household_id.isna().any() or assignment.household_id.duplicated().any():
    raise RuntimeError("Frozen household assignment IDs must be unique and non-null.")
intentional_holdout_ids = set(assignment.loc[assignment.holdout_group.eq("intentional_holdout"), "household_id"].astype(str))
if not intentional_holdout_ids:
    raise RuntimeError("No intentional household holdout IDs found.")
if not set(master.household_id.astype(str)).issubset(set(assignment.household_id.astype(str))):
    raise RuntimeError("Household assignment does not cover all master-split households.")

print("Provenance hashes and frozen policies verified.")
print(f"Intentional holdout households excluded in this experiment: {len(intentional_holdout_ids):,}.")
print("Frozen selected overall families:", {t: selected_models[t]["model_family"] for t in TARGETS})
print("Four per-family settings are loaded from 05_06_best_config_per_family.csv.")

## Stage 2 — Load eligible rows and verify class distributions

Eligible rows are the three internal development blocks (`model_development`, `calibration`, and `threshold_selection`) after excluding Notebook 04B intentional holdouts. No outer-test target/features are parsed. P95/P90 rows are joined to the master split by `row_id`; household, date, target, and feature checks must pass before modeling.

In [ ]:
def read_contract(path):
    return read_json(path)

def read_contract_csv(path, contract_path, usecols=None, skiprows=None):
    contract = read_contract(contract_path)
    header = pd.read_csv(path, nrows=0, encoding=contract["encoding"], sep=contract["separator"]).columns.tolist()
    if header != list(contract["columns"]):
        raise RuntimeError(f"CSV header differs from its dtype contract: {path.name}")
    columns = list(contract["columns"]) if usecols is None else list(usecols)
    dates = [name for name in contract.get("date_columns", []) if name in columns]
    dtypes = {name: dtype for name, dtype in contract["columns"].items()
              if name in columns and dtype != "datetime"}
    reader = contract["reader"]
    return pd.read_csv(path, usecols=columns, dtype=dtypes or None, parse_dates=dates or None,
        skiprows=skiprows, true_values=reader["true_values"], false_values=reader["false_values"],
        keep_default_na=reader["keep_default_na"], na_values=reader["na_values"],
        encoding=contract["encoding"], sep=contract["separator"], low_memory=False)

def read_only_expected_rows(path, contract_path, required_columns, expected_ids):
    """Scan row_id only, then parse required values for permitted row IDs only."""
    selected_line_numbers = set()
    found_ids = set()
    physical_line = 1
    for chunk in pd.read_csv(path, usecols=["row_id"], dtype={"row_id": "string"}, chunksize=100_000):
        ids = chunk["row_id"].astype(str).to_numpy()
        for offset, row_id in enumerate(ids):
            if row_id in expected_ids:
                selected_line_numbers.add(physical_line + offset)
                found_ids.add(row_id)
        physical_line += len(ids)
    if found_ids != expected_ids:
        raise RuntimeError(f"{path.name} row IDs differ from permitted frozen IDs: missing={len(expected_ids-found_ids)}, extra={len(found_ids-expected_ids)}")
    frame = read_contract_csv(path, contract_path, usecols=required_columns,
        skiprows=lambda line_number: line_number > 0 and line_number not in selected_line_numbers)
    if len(frame) != len(expected_ids) or frame.row_id.astype(str).duplicated().any():
        raise RuntimeError(f"Selective read returned the wrong number of unique rows for {path.name}.")
    return frame

permitted_master = master.loc[
    master.internal_split.isin(["model_development", "calibration", "threshold_selection"])
    & ~master.household_id.astype(str).isin(intentional_holdout_ids)
].copy()
if permitted_master.empty:
    raise RuntimeError("No eligible internal development rows remain after the frozen holdout exclusion.")
permitted_ids = set(permitted_master.row_id.astype(str))
permitted_master = permitted_master.set_index("row_id", drop=False)

COMMON_FEATURES = [
    "lag1_peak_kw", "lag2_peak_kw", "peak_d7", "peak_d14", "peak_d21", "peak_d28",
    "lag1_mean_kw", "lag7_mean_kw", "lag1_import_active_ratio", "lag7_import_active_ratio",
    "lag1_load_factor", "lag1_interval_p95_kw", "lag1_peak_hour_sin", "lag1_peak_hour_cos",
    "same_weekday_peak_mean4", "same_weekday_valid_count_4", "peak_mean_3d", "peak_mean_7d",
    "peak_mean_30d", "peak_max_7d", "peak_max_30d", "peak_std_7d", "peak_std_30d",
    "valid_peak_days_7d", "valid_peak_days_30d", "has_lag7", "peak_change_1d",
    "peak_mean3_over_30", "peak_mean_7d_over_30d", "day_of_week_sin", "day_of_week_cos",
    "month_sin", "month_cos", "is_weekend", "is_public_holiday", "is_poya_day",
]
FEATURES = {
    "P95": COMMON_FEATURES + ["lag1_peak_over_p95", "lag1_peak_minus_p95", "lag2_peak_over_p95",
        "lag7_peak_over_p95", "peak_mean_7d_over_p95", "p95_high_count_7d", "p95_high_count_30d",
        "p95_eligible_event_days_7d", "p95_eligible_event_days_30d", "p95_high_rate_30d", "p95_days_since_last_high"],
    "P90": COMMON_FEATURES + ["lag1_peak_over_p90", "lag1_peak_minus_p90", "lag2_peak_over_p90",
        "lag7_peak_over_p90", "peak_mean_7d_over_p90", "p90_high_count_7d", "p90_high_count_30d",
        "p90_eligible_event_days_7d", "p90_eligible_event_days_30d", "p90_high_rate_30d", "p90_days_since_last_high"],
}
for target in TARGETS:
    if FEATURES[target] != selected_models[target]["feature_list"]:
        raise RuntimeError(f"{target} feature list differs from the frozen Notebook 05 selected-model handoff.")
    forbidden = {"row_id", "household_id", "target", "measurement_date", "target_date", "outer_split", "internal_split"}
    if forbidden.intersection(FEATURES[target]):
        raise RuntimeError(f"Forbidden identifier/target columns found among {target} features.")

frames = {}
for target, path, contract_path in (
    ("P95", P95_PATH, P95_CONTRACT_PATH), ("P90", P90_PATH, P90_CONTRACT_PATH)):
    usecols = ["row_id", "household_id", "measurement_date", "target", *FEATURES[target]]
    frame = read_only_expected_rows(path, contract_path, usecols, permitted_ids)
    frame["row_id"] = frame.row_id.astype("string")
    frame["household_id"] = frame.household_id.astype("string")
    frame["measurement_date"] = pd.to_datetime(frame.measurement_date, errors="raise").dt.normalize()
    split_cols = permitted_master[["household_id", "target_date", "outer_split", "internal_split"]]
    frame = frame.merge(split_cols, left_on="row_id", right_index=True, how="left", validate="one_to_one", suffixes=("", "_split"))
    if frame.internal_split.isna().any() or frame.outer_split.ne("development").any():
        raise RuntimeError(f"{target} loaded rows outside internal development blocks.")
    if not frame.measurement_date.eq(frame.target_date).all():
        raise RuntimeError(f"{target} measurement dates disagree with frozen master split.")
    if not frame.household_id.astype(str).eq(frame.household_id_split.astype(str)).all():
        raise RuntimeError(f"{target} household IDs disagree with frozen master split.")
    frame = frame.drop(columns=["household_id_split"])
    if frame.target.isna().any() or not frame.target.isin([0, 1]).all():
        raise RuntimeError(f"{target} target is not complete binary 0/1.")
    if set(frame.row_id.astype(str)) != permitted_ids:
        raise RuntimeError(f"{target} loaded rows differ from eligible development IDs.")
    frames[target] = frame.sort_values(["measurement_date", "household_id"], kind="mergesort").reset_index(drop=True)

p95_keys = frames["P95"][["row_id", "household_id", "measurement_date", "internal_split"]].astype(str).reset_index(drop=True)
p90_keys = frames["P90"][["row_id", "household_id", "measurement_date", "internal_split"]].astype(str).reset_index(drop=True)
if not p95_keys.equals(p90_keys):
    raise RuntimeError("P95/P90 eligible row population or ordering differs.")

load_summary = pd.DataFrame([{
    "target": target, "eligible_rows": len(frame), "eligible_positive_events": int(frame.target.sum()),
    "eligible_prevalence": float(frame.target.mean()),
    "model_development_rows": int(frame.internal_split.eq("model_development").sum()),
    "calibration_rows": int(frame.internal_split.eq("calibration").sum()),
    "threshold_selection_rows": int(frame.internal_split.eq("threshold_selection").sum()),
    "date_start": frame.measurement_date.min(), "date_end": frame.measurement_date.max(),
} for target, frame in frames.items()])
display(load_summary)
load_summary.to_csv(TABLE_DIR / "00_eligible_population_summary.csv", index=False)
print("Target values were parsed only for eligible development-block row IDs; no outer-test labels were loaded.")

### Original class balance

These counts establish the natural event prevalence before sampling. The outer test is absent; the display covers only eligible development blocks.

In [ ]:
class_distribution = pd.concat([
    pd.DataFrame({"target": target, "class": ["Negative", "Positive"],
        "count": [(frame.target == 0).sum(), (frame.target == 1).sum()]})
    for target, frame in frames.items()
], ignore_index=True)
class_distribution["share"] = class_distribution.groupby("target")["count"].transform(lambda x: x / x.sum())
display(class_distribution)
class_distribution.to_csv(TABLE_DIR / "01_original_class_distribution.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (target, frame) in zip(axes, frames.items()):
    counts = frame.target.value_counts().reindex([0, 1], fill_value=0)
    ax.bar(["Negative", "Positive"], counts.values, color=["#637a91", "#e07a5f"])
    ax.set_title(f"{target}: natural development class counts")
    ax.set_ylabel("Rows")
    for i, value in enumerate(counts.values): ax.text(i, value, f"{value:,}\n({value/counts.sum():.1%})", ha="center", va="bottom")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "01_original_class_distribution.png", dpi=160, bbox_inches="tight")
plt.show()

## Stage 3 — Reconstruct and verify Notebook 05 chronological folds

Use the frozen date boundaries; do not make new folds after sampling. In each fold, training dates precede validation dates. The validation row IDs and class distribution stay fixed for every model and sampling strategy.

In [ ]:
frozen_folds = pd.read_csv(FOLD_PATH, parse_dates=["train_end", "validation_start", "validation_end"])
required_fold_columns = {"fold", "train_end", "validation_start", "validation_end", "train_rows", "validation_rows",
    "P95_train_events", "P95_validation_events", "P90_train_events", "P90_validation_events"}
if not required_fold_columns.issubset(frozen_folds.columns) or len(frozen_folds) != 3:
    raise RuntimeError("Expected the three frozen expanding folds from Notebook 05.")
model_development = {target: frame.loc[frame.internal_split.eq("model_development")].copy().reset_index(drop=True)
                     for target, frame in frames.items()}
folds = {}
fold_audit_rows = []
for row in frozen_folds.itertuples(index=False):
    fold_number = int(row.fold)
    p95_dev = model_development["P95"]
    train_mask = p95_dev.measurement_date.le(pd.Timestamp(row.train_end).normalize())
    valid_mask = p95_dev.measurement_date.gt(pd.Timestamp(row.train_end).normalize()) & p95_dev.measurement_date.le(pd.Timestamp(row.validation_end).normalize())
    if not train_mask.any() or not valid_mask.any():
        raise RuntimeError(f"Frozen fold {fold_number} has an empty training or validation block.")
    train_ids = set(p95_dev.loc[train_mask, "row_id"].astype(str))
    valid_ids = set(p95_dev.loc[valid_mask, "row_id"].astype(str))
    if train_ids & valid_ids:
        raise RuntimeError(f"Frozen fold {fold_number} training and validation IDs overlap.")
    fold_dict = {"fold": fold_number, "train_end": pd.Timestamp(row.train_end).normalize(),
        "validation_start": pd.Timestamp(row.validation_start).normalize(),
        "validation_end": pd.Timestamp(row.validation_end).normalize(),
        "train_ids": train_ids, "validation_ids": valid_ids}
    folds[fold_number] = fold_dict
    record = {"fold": fold_number, "train_end": fold_dict["train_end"],
        "validation_start": fold_dict["validation_start"], "validation_end": fold_dict["validation_end"],
        "train_rows": len(train_ids), "validation_rows": len(valid_ids)}
    if record["train_rows"] != int(row.train_rows) or record["validation_rows"] != int(row.validation_rows):
        raise RuntimeError(f"Reconstructed row counts differ from Notebook 05 fold {fold_number}.")
    for target in TARGETS:
        frame = model_development[target]
        if set(frame.row_id.astype(str)) != set(p95_dev.row_id.astype(str)):
            raise RuntimeError(f"{target} development row population differs from P95.")
        train = frame.loc[frame.row_id.astype(str).isin(train_ids)]
        valid = frame.loc[frame.row_id.astype(str).isin(valid_ids)]
        if train.empty or valid.empty or train.target.nunique() != 2 or valid.target.nunique() != 2:
            raise RuntimeError(f"{target} fold {fold_number} lacks both classes in training or validation.")
        if train.measurement_date.max() >= valid.measurement_date.min():
            raise RuntimeError(f"Temporal isolation failed in {target} fold {fold_number}.")
        record[f"{target}_train_events"] = int(train.target.sum())
        record[f"{target}_validation_events"] = int(valid.target.sum())
        record[f"{target}_validation_prevalence"] = float(valid.target.mean())
        if int(train.target.sum()) != int(getattr(row, f"{target}_train_events")) or int(valid.target.sum()) != int(getattr(row, f"{target}_validation_events")):
            raise RuntimeError(f"{target} event counts differ from frozen fold table in fold {fold_number}.")
    fold_audit_rows.append(record)
fold_audit = pd.DataFrame(fold_audit_rows).sort_values("fold")
display(fold_audit)
fold_audit.to_csv(TABLE_DIR / "02_cv_fold_audit.csv", index=False)
print("Confirmed: all validation periods are later than their training periods and match Notebook 05 row/event counts.")

## Stage 4 — Reproducible training-only undersampling

The requested ratio is **positive:negative**. For a 1:4 run, keep every positive training row and choose `4 × positives` negative training rows without replacement. Validation rows are never passed to the sampler. Sampled IDs are generated once per target/fold/ratio/seed and reused across all four algorithms.

In [ ]:
def random_undersample_ids(train_frame, ratio, seed):
    y = train_frame.target.to_numpy(dtype=np.int8)
    positive_ids = train_frame.loc[y == 1, "row_id"].astype(str).to_numpy()
    negative_ids = train_frame.loc[y == 0, "row_id"].astype(str).to_numpy()
    if len(positive_ids) == 0 or len(negative_ids) == 0:
        raise ValueError("Undersampling requires positive and negative training examples.")
    requested_negatives = int(np.floor(float(ratio) * len(positive_ids)))
    if requested_negatives < 1 or requested_negatives > len(negative_ids):
        raise ValueError(f"Requested {requested_negatives} negatives from {len(negative_ids)} available.")
    rng = np.random.default_rng(int(seed))
    sampled_negatives = rng.choice(negative_ids, size=requested_negatives, replace=False)
    keep = set(positive_ids) | set(sampled_negatives.astype(str))
    # Retain the original chronological/household ordering after drawing the random subset.
    selected = train_frame.loc[train_frame.row_id.astype(str).isin(keep), "row_id"].astype(str).tolist()
    if len(selected) != len(positive_ids) + requested_negatives:
        raise RuntimeError("Sampler returned duplicate or missing row IDs.")
    return selected

sampled_ids = {}
sampling_audit_rows = []
sampled_id_frames = []
for target in TARGETS:
    dev = model_development[target]
    for fold_number, fold in folds.items():
        train = dev.loc[dev.row_id.astype(str).isin(fold["train_ids"])].copy()
        validation_ids = fold["validation_ids"]
        for ratio_label, ratio_value in PRIMARY_RATIOS.items():
            for seed in SAMPLING_SEEDS:
                key = (target, fold_number, ratio_label, seed)
                selected_ids = random_undersample_ids(train, ratio_value, seed)
                if set(selected_ids) & validation_ids:
                    raise RuntimeError(f"Validation ID leaked into sampled training subset: {key}")
                sampled_ids[key] = selected_ids
                sampled = train.loc[train.row_id.astype(str).isin(set(selected_ids))]
                positives = int(sampled.target.sum())
                negatives = int((sampled.target == 0).sum())
                sampling_audit_rows.append({"target": target, "fold": fold_number, "sampling_strategy": ratio_label,
                    "positive_to_negative_ratio": ratio_label, "random_seed": seed,
                    "training_positive_count": positives, "training_negative_count": negatives,
                    "sampled_training_rows": len(sampled), "actual_negative_per_positive": negatives / positives,
                    "original_training_rows": len(train), "original_training_prevalence": float(train.target.mean()),
                    "validation_rows_unchanged": len(validation_ids)})
                sampled_id_frames.append(pd.DataFrame({
                    "target": target, "fold": fold_number, "sampling_strategy": ratio_label,
                    "random_seed": seed, "row_id": selected_ids,
                }))

sampling_audit = pd.DataFrame(sampling_audit_rows)
sampled_training_ids = pd.concat(sampled_id_frames, ignore_index=True)
sampling_audit.to_csv(TABLE_DIR / "03_undersampling_audit.csv", index=False)
sampled_training_ids.to_csv(TABLE_DIR / "04_sampled_training_row_ids.csv", index=False)
print(f"Saved {len(sampling_audit):,} sampling audit rows and {len(sampled_training_ids):,} reproducible sampled row-ID assignments.")
display(sampling_audit.head(12))

In [ ]:
# Check event retention, realized class ratios, and calendar coverage after sampling.
if not sampling_audit.training_positive_count.eq(sampling_audit.groupby(["target", "fold"]).training_positive_count.transform("first")).all():
    raise RuntimeError("A sampling strategy did not retain every positive example in its fold.")
if not sampling_audit.actual_negative_per_positive.le(sampling_audit.positive_to_negative_ratio.str.split(":").str[1].astype(float)).all():
    raise RuntimeError("A sampled negative:positive ratio exceeded its requested maximum.")

calendar_coverage_rows = []
for row in sampling_audit.itertuples(index=False):
    key = (row.target, int(row.fold), row.sampling_strategy, int(row.random_seed))
    ids = set(sampled_ids[key])
    train = model_development[row.target].loc[model_development[row.target].row_id.astype(str).isin(ids)]
    for month, part in train.assign(month=train.measurement_date.dt.to_period("M").astype(str)).groupby("month"):
        calendar_coverage_rows.append({"target": row.target, "fold": row.fold,
            "sampling_strategy": row.sampling_strategy, "random_seed": row.random_seed,
            "month": month, "sampled_rows": len(part), "sampled_positive_events": int(part.target.sum())})
calendar_coverage = pd.DataFrame(calendar_coverage_rows)
calendar_coverage.to_csv(TABLE_DIR / "04b_sampled_calendar_coverage.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
for ax, target in zip(axes, TARGETS):
    subset = sampling_audit.loc[(sampling_audit.target == target) & (sampling_audit.fold == 3)]
    means = subset.groupby("sampling_strategy")[["training_positive_count", "training_negative_count"]].mean().reindex(PRIMARY_RATIOS)
    means.plot(kind="bar", ax=ax, color=["#e07a5f", "#637a91"])
    ax.set_title(f"{target}: Fold 3 sampled training counts")
    ax.set_xlabel("Positive:negative strategy")
    ax.set_ylabel("Training rows")
    ax.legend(["Positive", "Negative"])
fig.tight_layout()
fig.savefig(FIGURE_DIR / "02_sampled_training_class_counts.png", dpi=160, bbox_inches="tight")
plt.show()

display(calendar_coverage.groupby(["target", "sampling_strategy", "month"], as_index=False).sampled_rows.mean().head(24))
print("Calendar coverage is audited by month. The sampler is random overall; it does not balance by household, branch, or outcome-derived subgroup.")

## Shared training and scoring helpers

Preprocessing is fitted inside each estimator using only its current training subset. Model hyperparameters come from Notebook 05's best configuration for that target and family. Undersampled runs disable class weighting so the comparison isolates random undersampling. The original frozen configuration remains a separate benchmark.

In [ ]:
def numeric_x(frame, features):
    return frame.loc[:, features].apply(lambda col: pd.to_numeric(col, errors="coerce").astype("float64"))

def without_imbalance_weight(family, params):
    params = dict(params)
    if family in ("Logistic Regression", "Random Forest"):
        params["class_weight"] = None
    elif family == "XGBoost":
        params["scale_pos_weight_mode"] = "one"
    elif family == "CatBoost":
        params["auto_class_weights"] = None
    return params

def build_estimator(family, params, y_fit):
    params = dict(params)
    imputer = SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)
    if family == "Logistic Regression":
        model = LogisticRegression(C=float(params["C"]), penalty="l2", class_weight=params.get("class_weight"),
            solver="liblinear", max_iter=2000, random_state=MODEL_RANDOM_STATE)
        return Pipeline([("imputer", imputer), ("scaler", StandardScaler()), ("model", model)])
    if family == "Random Forest":
        model = RandomForestClassifier(n_estimators=int(params["n_estimators"]), max_depth=params.get("max_depth"),
            min_samples_leaf=int(params["min_samples_leaf"]), max_features=params["max_features"],
            class_weight=params.get("class_weight"), n_jobs=N_JOBS, random_state=MODEL_RANDOM_STATE)
        return Pipeline([("imputer", imputer), ("model", model)])
    if family == "XGBoost":
        mode = params.pop("scale_pos_weight_mode", "one")
        ratio = float((np.asarray(y_fit) == 0).sum() / max((np.asarray(y_fit) == 1).sum(), 1))
        weights = {"one": 1.0, "sqrt_ratio": float(np.sqrt(ratio)), "half_ratio": ratio / 2.0, "full_ratio": ratio}
        if mode not in weights:
            raise ValueError(f"Unknown frozen XGBoost weighting mode: {mode}")
        model = XGBClassifier(n_estimators=int(params["n_estimators"]), max_depth=int(params["max_depth"]),
            learning_rate=float(params["learning_rate"]), min_child_weight=float(params["min_child_weight"]),
            subsample=float(params["subsample"]), colsample_bytree=float(params["colsample_bytree"]),
            reg_alpha=float(params["reg_alpha"]), reg_lambda=float(params["reg_lambda"]),
            scale_pos_weight=weights[mode], objective="binary:logistic", eval_metric="logloss",
            tree_method="hist", n_jobs=N_JOBS, random_state=MODEL_RANDOM_STATE, verbosity=0)
        return Pipeline([("imputer", imputer), ("model", model)])
    if family == "CatBoost":
        model = CatBoostClassifier(iterations=int(params["iterations"]), depth=int(params["depth"]),
            learning_rate=float(params["learning_rate"]), l2_leaf_reg=float(params["l2_leaf_reg"]),
            auto_class_weights=params.get("auto_class_weights"), loss_function="Logloss", eval_metric="Logloss",
            random_seed=MODEL_RANDOM_STATE, verbose=False, allow_writing_files=False, thread_count=N_JOBS)
        return Pipeline([("imputer", imputer), ("model", model)])
    raise KeyError(family)

def model_scenarios(target, family):
    frozen = dict(family_params[(target, family)])
    return [
        ("Original / unweighted", without_imbalance_weight(family, frozen), None),
        ("Original / frozen Notebook 05 config", frozen, None),
        *[(label, without_imbalance_weight(family, frozen), ratio)
          for label, ratio in PRIMARY_RATIOS.items()],
    ]

def score_model(family, params, train, valid, target):
    features = FEATURES[target]
    y_train = train.target.to_numpy(dtype=np.int8)
    y_valid = valid.target.to_numpy(dtype=np.int8)
    estimator = build_estimator(family, params, y_train)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)
        estimator.fit(numeric_x(train, features), y_train)
    probability = estimator.predict_proba(numeric_x(valid, features))[:, 1]
    if len(probability) != len(valid) or not np.isfinite(probability).all():
        raise RuntimeError(f"{family} produced invalid probabilities for {target}.")
    predicted = (probability >= 0.5).astype(np.int8)
    return {
        "pr_auc": float(average_precision_score(y_valid, probability)),
        "roc_auc": float(roc_auc_score(y_valid, probability)),
        "precision_at_0_5": float(precision_score(y_valid, predicted, zero_division=0)),
        "recall_at_0_5": float(recall_score(y_valid, predicted, zero_division=0)),
        "f1_at_0_5": float(f1_score(y_valid, predicted, zero_division=0)),
        "balanced_accuracy_at_0_5": float(balanced_accuracy_score(y_valid, predicted)),
    }

def run_family(family):
    filename = {"Logistic Regression": "05_logistic_regression_results.csv",
        "Random Forest": "06_random_forest_results.csv", "XGBoost": "07_xgboost_results.csv",
        "CatBoost": "08_catboost_results.csv"}[family]
    result_path = TABLE_DIR / filename
    if result_path.exists():
        cached = pd.read_csv(result_path)
        expected_keys = set()
        expected_params = {}
        for target in TARGETS:
            for strategy, params, ratio in model_scenarios(target, family):
                expected_params[(target, strategy)] = json.dumps(params, sort_keys=True)
                for fold_number in folds:
                    seeds = SAMPLING_SEEDS if ratio is not None else (MODEL_RANDOM_STATE,)
                    for seed in seeds:
                        expected_keys.add((target, strategy, int(fold_number), int(seed)))
        actual_keys = set(zip(cached.target.astype(str), cached.sampling_strategy.astype(str),
            cached.cv_fold.astype(int), cached.random_seed.astype(int))) if not cached.empty else set()
        required_metrics = ["pr_auc", "roc_auc", "precision_at_0_5", "recall_at_0_5", "f1_at_0_5", "balanced_accuracy_at_0_5"]
        cache_valid = (len(cached) == len(expected_keys) and not cached.duplicated(
            ["target", "sampling_strategy", "cv_fold", "random_seed"]).any()
            and actual_keys == expected_keys and cached.model_family.eq(family).all()
            and np.isfinite(cached[required_metrics].to_numpy(dtype=float)).all())
        if cache_valid:
            for row in cached.itertuples(index=False):
                if json.dumps(json.loads(row.model_parameters_json), sort_keys=True) != expected_params[(row.target, row.sampling_strategy)]:
                    cache_valid = False
                    break
        if cache_valid:
            print(f"{family}: reusing the complete, validated result table {filename}.")
            return cached
        print(f"{family}: existing result table is incomplete or incompatible; recomputing.")
    rows = []
    total = 0
    for target in TARGETS:
        for strategy, params, ratio in model_scenarios(target, family):
            for fold_number, fold in folds.items():
                dev = model_development[target]
                train = dev.loc[dev.row_id.astype(str).isin(fold["train_ids"])].copy()
                valid = dev.loc[dev.row_id.astype(str).isin(fold["validation_ids"])].copy()
                seeds = SAMPLING_SEEDS if ratio is not None else (MODEL_RANDOM_STATE,)
                for seed in seeds:
                    if ratio is None:
                        fit = train
                    else:
                        ratio_label = strategy
                        ids = sampled_ids[(target, fold_number, ratio_label, int(seed))]
                        fit = train.loc[train.row_id.astype(str).isin(set(ids))].copy()
                    metrics = score_model(family, params, fit, valid, target)
                    positives = int(fit.target.sum())
                    negatives = int((fit.target == 0).sum())
                    rows.append({"target": target, "model_family": family, "sampling_strategy": strategy,
                        "positive_to_negative_ratio": strategy.split(" ")[-1] if ratio is not None else "Original",
                        "random_seed": int(seed), "cv_fold": int(fold_number),
                        "model_parameters_json": json.dumps(params, sort_keys=True),
                        "training_rows": len(fit), "training_positive_count": positives,
                        "training_negative_count": negatives, "validation_rows": len(valid),
                        "validation_positive_count": int(valid.target.sum()),
                        "validation_prevalence": float(valid.target.mean()), **metrics})
                    total += 1
                    if total % 12 == 0:
                        print(f"  {family}: completed {total} fits")
    result = pd.DataFrame(rows)
    if result.empty or result.duplicated(["target", "sampling_strategy", "cv_fold", "random_seed"]).any():
        raise RuntimeError(f"{family} result table is empty or has duplicate runs.")
    result.to_csv(result_path, index=False, float_format="%.17g")
    return result

def plot_family_results(result, family, filename):
    grouped = result.groupby(["target", "sampling_strategy"], as_index=False).agg(mean_pr_auc=("pr_auc", "mean"), std_pr_auc=("pr_auc", "std"))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    order = ["Original / unweighted", "Original / frozen Notebook 05 config", *PRIMARY_RATIOS.keys()]
    for ax, target in zip(axes, TARGETS):
        part = grouped.loc[grouped.target.eq(target)].set_index("sampling_strategy").reindex(order).dropna().reset_index()
        ax.bar(part.sampling_strategy, part.mean_pr_auc, yerr=part.std_pr_auc.fillna(0), capsize=3, color="#3d7a8c")
        ax.set_title(f"{target}: {family}")
        ax.set_ylabel("Mean validation PR-AUC")
        ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / filename, dpi=160, bbox_inches="tight")
    plt.show()

## Model 1 — Logistic Regression

A linear baseline. The imputer and scaler are fitted only on the current training fold; the model itself uses the frozen best `C` and disables class weighting in the original-unweighted and undersampled-only comparisons.

In [ ]:
logistic_results = run_family("Logistic Regression")
display(logistic_results.groupby(["target", "sampling_strategy"], as_index=False).agg(mean_pr_auc=("pr_auc", "mean"), std_pr_auc=("pr_auc", "std"), runs=("pr_auc", "count")))

In [ ]:
plot_family_results(logistic_results, "Logistic Regression", "03_logistic_regression_pr_auc.png")

## Model 2 — Random Forest

A nonlinear tree baseline. Tree settings stay at the frozen best-per-family values. The undersampled-only runs set `class_weight=None`; the separately reported frozen-configuration benchmark keeps Notebook 05's original setting.

In [ ]:
random_forest_results = run_family("Random Forest")
display(random_forest_results.groupby(["target", "sampling_strategy"], as_index=False).agg(mean_pr_auc=("pr_auc", "mean"), std_pr_auc=("pr_auc", "std"), runs=("pr_auc", "count")))

In [ ]:
plot_family_results(random_forest_results, "Random Forest", "04_random_forest_pr_auc.png")

## Model 3 — XGBoost

XGBoost was the overall selected P95 family in Notebook 05. Undersampled-only and original-unweighted runs set `scale_pos_weight=1`; the frozen Notebook 05 comparison retains its saved weighting mode. The selected tree and regularization settings do not change across ratios.

In [ ]:
xgboost_results = run_family("XGBoost")
display(xgboost_results.groupby(["target", "sampling_strategy"], as_index=False).agg(mean_pr_auc=("pr_auc", "mean"), std_pr_auc=("pr_auc", "std"), runs=("pr_auc", "count")))

In [ ]:
plot_family_results(xgboost_results, "XGBoost", "05_xgboost_pr_auc.png")

## Model 4 — CatBoost

CatBoost is the fourth comparator. The pipeline fits the imputer on training data, disables CatBoost's automatic class weights for undersampled-only runs, and prevents CatBoost from writing training files outside this experiment.

In [ ]:
catboost_results = run_family("CatBoost")
display(catboost_results.groupby(["target", "sampling_strategy"], as_index=False).agg(mean_pr_auc=("pr_auc", "mean"), std_pr_auc=("pr_auc", "std"), runs=("pr_auc", "count")))

In [ ]:
plot_family_results(catboost_results, "CatBoost", "06_catboost_pr_auc.png")

## Stage 5 — Consolidated model comparison

Precision, recall, F1, and balanced accuracy below use a diagnostic threshold of 0.5. Because undersampling can shift predicted probabilities, these are not final operating-point claims. Compare PR-AUC first; use the later natural-prevalence calibration and threshold stage for a development-only operating threshold.

In [ ]:
all_results = pd.concat([logistic_results, random_forest_results, xgboost_results, catboost_results], ignore_index=True)
all_results.to_csv(TABLE_DIR / "09_all_model_comparison.csv", index=False, float_format="%.17g")
metric_cols = ["pr_auc", "roc_auc", "precision_at_0_5", "recall_at_0_5", "f1_at_0_5", "balanced_accuracy_at_0_5"]
comparison = all_results.groupby(["target", "model_family", "sampling_strategy"], as_index=False).agg(
    **{f"mean_{col}": (col, "mean") for col in metric_cols},
    **{f"std_{col}": (col, "std") for col in metric_cols},
    fit_count=("cv_fold", "count"),
    mean_training_rows=("training_rows", "mean"),
)
comparison = comparison.sort_values(["target", "mean_pr_auc", "std_pr_auc"], ascending=[True, False, True])
display(comparison)
comparison.to_csv(TABLE_DIR / "09a_consolidated_mean_comparison.csv", index=False, float_format="%.17g")
stability = all_results.groupby(["target", "model_family", "sampling_strategy"], as_index=False).agg(
    mean_pr_auc=("pr_auc", "mean"), std_pr_auc=("pr_auc", "std"), min_pr_auc=("pr_auc", "min"),
    max_pr_auc=("pr_auc", "max"), fold_seed_runs=("pr_auc", "count"))
stability.to_csv(TABLE_DIR / "10_sampling_stability.csv", index=False, float_format="%.17g")

In [ ]:
# Figure 1: mean PR-AUC across family and strategy, separately for P95 and P90.
strategy_order = ["Original / unweighted", "Original / frozen Notebook 05 config", *PRIMARY_RATIOS.keys()]
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, target in zip(axes, TARGETS):
    part = comparison.loc[comparison.target.eq(target)].copy()
    pivot = part.pivot(index="model_family", columns="sampling_strategy", values="mean_pr_auc")
    pivot = pivot.reindex(index=FAMILIES, columns=strategy_order)
    pivot.plot(kind="bar", ax=ax, color=["#829399", "#d9a441", "#79a9a3", "#558b8d", "#376b72"])
    ax.set_title(f"{target}: validation PR-AUC by model and strategy")
    ax.set_ylabel("Mean PR-AUC")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=18)
    ax.legend(title="Training strategy", fontsize=8)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "07_pr_auc_model_sampling_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

# Figure 2: change from original unweighted model within each family.
base = comparison.loc[comparison.sampling_strategy.eq("Original / unweighted"),
    ["target", "model_family", "mean_pr_auc"]].rename(columns={"mean_pr_auc": "baseline_pr_auc"})
delta = comparison.merge(base, on=["target", "model_family"], how="left", validate="many_to_one")
delta["delta_pr_auc"] = delta.mean_pr_auc - delta.baseline_pr_auc
delta.to_csv(TABLE_DIR / "09b_pr_auc_change_from_unweighted.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, target in zip(axes, TARGETS):
    part = delta.loc[(delta.target.eq(target)) & ~delta.sampling_strategy.eq("Original / unweighted")]
    for family in FAMILIES:
        line = part.loc[part.model_family.eq(family)]
        ax.plot(line.sampling_strategy, line.delta_pr_auc, marker="o", label=family)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title(f"{target}: mean PR-AUC change vs original unweighted")
    ax.set_ylabel("Δ PR-AUC")
    ax.tick_params(axis="x", rotation=25)
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "08_delta_pr_auc_vs_unweighted.png", dpi=160, bbox_inches="tight")
plt.show()

# Figure 3: diagnostic precision-recall points at threshold 0.5.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, target in zip(axes, TARGETS):
    part = comparison.loc[comparison.target.eq(target)]
    for family in FAMILIES:
        rows = part.loc[part.model_family.eq(family)]
        ax.scatter(rows.mean_precision_at_0_5, rows.mean_recall_at_0_5, label=family, alpha=.8)
    ax.set_title(f"{target}: precision vs recall at threshold 0.5")
    ax.set_xlabel("Mean precision")
    ax.set_ylabel("Mean recall")
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "09_precision_recall_diagnostic.png", dpi=160, bbox_inches="tight")
plt.show()

# Figure 4: every fold/seed PR-AUC point reveals stability.
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, target in zip(axes, TARGETS):
    part = all_results.loc[all_results.target.eq(target)]
    labels = [f"{f}\n{s}" for f in FAMILIES for s in strategy_order]
    data = [part.loc[(part.model_family.eq(f)) & (part.sampling_strategy.eq(s)), "pr_auc"].to_numpy()
            for f in FAMILIES for s in strategy_order]
    keep = [(lab, values) for lab, values in zip(labels, data) if len(values)]
    ax.boxplot([x[1] for x in keep], tick_labels=[x[0] for x in keep], showmeans=True)
    ax.set_title(f"{target}: fold/seed PR-AUC stability")
    ax.set_ylabel("Validation PR-AUC")
    ax.tick_params(axis="x", labelsize=7, rotation=45)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "10_fold_seed_stability.png", dpi=160, bbox_inches="tight")
plt.show()

## Stage 6 — Select development-only candidates

Rank each target's model/strategy by mean PR-AUC across its available folds and sampling seeds. Fold/seed variation breaks close ties. The already-evaluated outer test is not considered. It is valid for the original unweighted or frozen Notebook 05 configuration to remain the winner.

In [ ]:
candidate_summary = all_results.groupby(["target", "model_family", "sampling_strategy"], as_index=False).agg(
    mean_cv_pr_auc=("pr_auc", "mean"), std_cv_pr_auc=("pr_auc", "std"),
    mean_cv_roc_auc=("roc_auc", "mean"), run_count=("pr_auc", "count"),
    mean_training_rows=("training_rows", "mean"))
candidate_summary["std_cv_pr_auc"] = candidate_summary.std_cv_pr_auc.fillna(0.0)
selected_candidates = {}
for target in TARGETS:
    ranked = candidate_summary.loc[candidate_summary.target.eq(target)].sort_values(
        ["mean_cv_pr_auc", "std_cv_pr_auc", "model_family"], ascending=[False, True, True], kind="stable")
    if ranked.empty:
        raise RuntimeError(f"No CV candidates available for {target}.")
    selected_candidates[target] = ranked.iloc[0].to_dict()
selected_candidate_table = pd.DataFrame(selected_candidates.values())
selected_candidate_table.to_csv(TABLE_DIR / "12_selected_development_candidates.csv", index=False)
display(selected_candidate_table)
print("Candidate selection used only the three model-development CV folds; the outer test was not accessed.")

## Stage 7 — Refit and calibrate the selected development candidates

Refit each selected candidate on the eligible `model_development` block. If undersampling won, resample only that base-model fit. Fit Platt calibration on the untouched natural-prevalence `calibration` block and choose an F1 threshold on `threshold_selection`. Exclude intentional holdouts consistently from all three stages. These threshold-selection metrics are development diagnostics, not independent final-test evidence.

In [ ]:
def probability_logit(probability):
    p = np.clip(np.asarray(probability, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p)).reshape(-1, 1)

def best_threshold(y_true, probability):
    rows = []
    for threshold in THRESHOLD_GRID:
        pred = (probability >= threshold).astype(np.int8)
        rows.append({"threshold": float(threshold),
            "precision": float(precision_score(y_true, pred, zero_division=0)),
            "recall": float(recall_score(y_true, pred, zero_division=0)),
            "f1": float(f1_score(y_true, pred, zero_division=0))})
    curve = pd.DataFrame(rows)
    chosen = curve.sort_values(["f1", "precision", "threshold"], ascending=[False, False, False], kind="stable").iloc[0]
    return curve, chosen

calibration_outputs = {}
calibration_rows = []
threshold_curve_rows = []
for target in TARGETS:
    candidate = selected_candidates[target]
    family = candidate["model_family"]
    strategy = candidate["sampling_strategy"]
    frozen = dict(family_params[(target, family)])
    params = frozen if strategy == "Original / frozen Notebook 05 config" else without_imbalance_weight(family, frozen)
    fit = frames[target].loc[frames[target].internal_split.eq("model_development")].copy()
    cal = frames[target].loc[frames[target].internal_split.eq("calibration")].copy()
    threshold = frames[target].loc[frames[target].internal_split.eq("threshold_selection")].copy()
    if fit.empty or cal.empty or threshold.empty:
        raise RuntimeError(f"{target}: calibration-stage development block is empty.")
    fit_seed = MODEL_RANDOM_STATE
    ratio_value = PRIMARY_RATIOS.get(strategy)
    if ratio_value is not None:
        fit_ids = random_undersample_ids(fit, ratio_value, fit_seed)
        fit = fit.loc[fit.row_id.astype(str).isin(set(fit_ids))].copy()
    y_fit = fit.target.to_numpy(dtype=np.int8)
    estimator = build_estimator(family, params, y_fit)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)
        estimator.fit(numeric_x(fit, FEATURES[target]), y_fit)
    raw_cal = estimator.predict_proba(numeric_x(cal, FEATURES[target]))[:, 1]
    raw_threshold = estimator.predict_proba(numeric_x(threshold, FEATURES[target]))[:, 1]
    calibrator = LogisticRegression(C=1.0, solver="lbfgs", max_iter=2000, random_state=MODEL_RANDOM_STATE)
    calibrator.fit(probability_logit(raw_cal), cal.target.to_numpy(dtype=np.int8))
    calibrated_threshold = calibrator.predict_proba(probability_logit(raw_threshold))[:, 1]
    curve, chosen = best_threshold(threshold.target.to_numpy(dtype=np.int8), calibrated_threshold)
    curve.insert(0, "target", target)
    curve.insert(1, "model_family", family)
    curve.insert(2, "sampling_strategy", strategy)
    threshold_curve_rows.append(curve)
    raw_brier = brier_score_loss(threshold.target, raw_threshold)
    calibrated_brier = brier_score_loss(threshold.target, calibrated_threshold)
    raw_logloss = log_loss(threshold.target, raw_threshold, labels=[0, 1])
    calibrated_logloss = log_loss(threshold.target, calibrated_threshold, labels=[0, 1])
    calibration_rows.append({"target": target, "model_family": family, "sampling_strategy": strategy,
        "mean_cv_pr_auc": candidate["mean_cv_pr_auc"], "fit_rows_after_sampling": len(fit),
        "fit_positive_count": int(fit.target.sum()), "fit_negative_count": int((fit.target == 0).sum()),
        "calibration_rows": len(cal), "threshold_selection_rows": len(threshold),
        "threshold_selection_positive_count": int(threshold.target.sum()),
        "raw_brier_score_threshold_selection": float(raw_brier),
        "calibrated_brier_score_threshold_selection": float(calibrated_brier),
        "raw_log_loss_threshold_selection": float(raw_logloss),
        "calibrated_log_loss_threshold_selection": float(calibrated_logloss),
        "selected_threshold": float(chosen.threshold), "threshold_precision": float(chosen.precision),
        "threshold_recall": float(chosen.recall), "threshold_f1": float(chosen.f1),
        "threshold_rule": "maximize F1 on threshold_selection; ties by precision, then higher threshold"})
    calibration_outputs[target] = {"family": family, "strategy": strategy, "fit": fit,
        "cal": cal, "threshold": threshold, "estimator": estimator, "calibrator": calibrator,
        "raw_threshold": raw_threshold, "calibrated_threshold": calibrated_threshold,
        "chosen_threshold": float(chosen.threshold), "curve": curve}
    safe_family = family.lower().replace(" ", "_")
    bundle_path = MODEL_DIR / f"{target.lower()}_{safe_family}_undersampling_experiment.joblib"
    joblib.dump({"target": target, "model_family": family, "sampling_strategy": strategy,
        "model_parameters": params, "features": FEATURES[target], "base_estimator": estimator,
        "platt_calibrator": calibrator, "threshold": float(chosen.threshold),
        "threshold_rule": "maximize F1 on threshold_selection; ties by precision, then higher threshold",
        "training_scope": "eligible internal development; intentional holdout households excluded",
        "outer_test_used": False}, bundle_path)
    print(f"{target}: selected {family} / {strategy}; threshold={chosen.threshold:.2f}; saved isolated bundle {bundle_path.relative_to(ROOT)}")

calibration_comparison = pd.DataFrame(calibration_rows)
threshold_curves = pd.concat(threshold_curve_rows, ignore_index=True)
calibration_comparison.to_csv(TABLE_DIR / "11_calibration_comparison.csv", index=False, float_format="%.17g")
threshold_curves.to_csv(TABLE_DIR / "11b_threshold_search_curves.csv", index=False, float_format="%.17g")
display(calibration_comparison)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for row_idx, target in enumerate(TARGETS):
    output = calibration_outputs[target]
    y = output["threshold"].target.to_numpy(dtype=np.int8)
    for col_idx, (label, probabilities) in enumerate([
        ("Raw", output["raw_threshold"]), ("Platt calibrated", output["calibrated_threshold"])]):
        ax = axes[row_idx, col_idx]
        observed, predicted = calibration_curve(y, probabilities, n_bins=10, strategy="quantile")
        ax.plot(predicted, observed, marker="o", label=label)
        ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Ideal")
        ax.set_title(f"{target}: {label} probabilities")
        ax.set_xlabel("Mean predicted probability")
        ax.set_ylabel("Observed event rate")
        ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "11_calibration_reliability.png", dpi=160, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, target in zip(axes, TARGETS):
    output = calibration_outputs[target]
    curve = output["curve"]
    ax.plot(curve.threshold, curve.precision, label="Precision")
    ax.plot(curve.threshold, curve.recall, label="Recall")
    ax.plot(curve.threshold, curve.f1, label="F1")
    ax.axvline(output["chosen_threshold"], color="black", linestyle="--", label=f"Selected {output['chosen_threshold']:.2f}")
    ax.set_title(f"{target}: threshold-selection block")
    ax.set_xlabel("Probability threshold")
    ax.set_ylabel("Metric")
    ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "12_threshold_selection_curves.png", dpi=160, bbox_inches="tight")
plt.show()
print("Calibration plots use the natural-prevalence threshold-selection block. The reported operating metrics are development-only and not outer-test estimates.")

## Stage 8 — Findings summary and final audit

Generate conclusions from the executed results. No finding is prewritten. The final audit verifies the holdout exclusion, sampled-row provenance, unchanged validation IDs, unchanged source hashes, and absence of outer-test access.

In [ ]:
summary_rows = []
for target in TARGETS:
    candidate = selected_candidates[target]
    family = candidate["model_family"]
    strategy = candidate["sampling_strategy"]
    family_rows = candidate_summary.loc[candidate_summary.target.eq(target)]
    baseline = family_rows.loc[(family_rows.model_family.eq(family)) &
        (family_rows.sampling_strategy.eq("Original / unweighted"))]
    baseline_auc = float(baseline.iloc[0].mean_cv_pr_auc) if not baseline.empty else np.nan
    baseline_std = float(baseline.iloc[0].std_cv_pr_auc) if not baseline.empty else np.nan
    delta_auc = float(candidate["mean_cv_pr_auc"] - baseline_auc) if np.isfinite(baseline_auc) else np.nan
    observed_variation = max(float(candidate["std_cv_pr_auc"]), baseline_std if np.isfinite(baseline_std) else 0.0)
    if strategy.startswith("Original"):
        conclusion = "An original-data configuration ranked highest; undersampling did not improve the best development CV candidate."
    elif delta_auc > 0 and delta_auc <= observed_variation:
        conclusion = "The selected undersampling candidate has a small positive mean PR-AUC difference, smaller than the observed fold/seed variation; this is not clear evidence of a reliable improvement."
    elif delta_auc > 0:
        conclusion = "The selected undersampling candidate exceeded its same-family original-unweighted baseline by more than the observed fold/seed variation; confirm on independent future data before making a final-test claim."
    else:
        conclusion = "The selected candidate did not exceed its same-family original-unweighted baseline; inspect variability and class-threshold trade-offs."
    summary_rows.append({"target": target, "best_model_family": family, "best_sampling_strategy": strategy,
        "best_mean_cv_pr_auc": float(candidate["mean_cv_pr_auc"]),
        "same_family_original_unweighted_mean_cv_pr_auc": baseline_auc,
        "same_family_original_unweighted_std_cv_pr_auc": baseline_std,
        "delta_vs_same_family_unweighted": delta_auc,
        "cv_std_pr_auc": float(candidate["std_cv_pr_auc"]), "cv_runs": int(candidate["run_count"]),
        "calibrated_brier_score_threshold_selection": float(calibration_comparison.loc[calibration_comparison.target.eq(target), "calibrated_brier_score_threshold_selection"].iloc[0]),
        "development_only_conclusion": conclusion})
experiment_summary = pd.DataFrame(summary_rows)
experiment_summary.to_csv(TABLE_DIR / "12_experiment_summary.csv", index=False, float_format="%.17g")
display(experiment_summary)
for row in experiment_summary.itertuples(index=False):
    print(f"{row.target}: {row.development_only_conclusion}")
print("No outer-test result is claimed by this experiment.")

In [ ]:
# Final row-level and artifact audit.
checks = []
def add_check(name, passed, detail):
    checks.append({"check": name, "passed": bool(passed), "detail": str(detail)})

add_check("eligible_frames_contain_no_outer_test", all(frame.outer_split.eq("development").all() for frame in frames.values()),
          "Only model_development, calibration, threshold_selection internal blocks were loaded.")
add_check("intentional_holdout_excluded_from_all_loaded_rows",
          all(not set(frame.household_id.astype(str)).intersection(intentional_holdout_ids) for frame in frames.values()),
          f"Excluded {len(intentional_holdout_ids)} frozen intentional holdout households.")
add_check("frozen_fold_row_counts_reproduced",
          fold_audit.train_rows.tolist() == frozen_folds.train_rows.tolist() and fold_audit.validation_rows.tolist() == frozen_folds.validation_rows.tolist(),
          "Reconstructed from frozen date boundaries and eligible non-holdout model-development rows.")
add_check("sampled_ids_are_training_only", all(not (set(ids) & folds[key[1]]["validation_ids"]) for key, ids in sampled_ids.items()),
          f"Checked {len(sampled_ids)} target/fold/ratio/seed subsets.")
add_check("same_sampled_subsets_reused_across_model_families", len(sampled_ids) == len(TARGETS) * len(folds) * len(PRIMARY_RATIOS) * len(SAMPLING_SEEDS),
          "A single assignment map is consumed by each model-family section.")
add_check("validation_population_unchanged", all(len(folds[k]["validation_ids"]) == int(frozen_folds.loc[frozen_folds.fold.eq(k), "validation_rows"].iloc[0]) for k in folds),
          "Original naturally imbalanced validation row IDs are shared by all runs.")
add_check("all_targets_models_and_strategies_have_results",
          set(all_results.target) == set(TARGETS) and set(all_results.model_family) == set(FAMILIES) and set(all_results.sampling_strategy) >= {"Original / unweighted", "Original / frozen Notebook 05 config", *PRIMARY_RATIOS.keys()},
          f"{len(all_results):,} fold/seed result rows.")
current_hashes = {str(p.relative_to(ROOT)): sha256(p) for p in REQUIRED_INPUTS}
add_check("all_frozen_source_hashes_unchanged", source_hashes == current_hashes, "No frozen input was modified during execution.")
output_files = sorted(str(p.relative_to(ROOT)) for p in OUTPUT_DIR.rglob("*") if p.is_file())
add_check("outputs_confined_to_experiment_directory", all((ROOT / p).resolve().is_relative_to(OUTPUT_DIR.resolve()) for p in output_files),
          f"{len(output_files)} output files under {OUTPUT_DIR.relative_to(ROOT)}.")
add_check("no_outer_test_targets_used", True, "The selective loader parsed only the permitted internal-development row IDs.")

final_audit = pd.DataFrame(checks)
final_audit.to_csv(TABLE_DIR / "13_final_validation.csv", index=False)
if not final_audit.passed.all():
    display(final_audit.loc[~final_audit.passed])
    raise RuntimeError("At least one final experiment audit failed.")

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "experiment": "independent_random_undersampling_experiment",
    "notebook": "notebooks/experiments/class_imbalance/05c_random_undersampling_experiment.ipynb",
    "purpose": "Development-only random undersampling sensitivity experiment for rare household-relative high-demand events.",
    "primary_target": "P95", "secondary_target": "P90", "model_families": list(FAMILIES),
    "primary_sampling_ratios_positive_to_negative": PRIMARY_RATIOS,
    "sampling_seeds": list(SAMPLING_SEEDS), "model_random_state": MODEL_RANDOM_STATE,
    "cv_folds": 3, "primary_metric": "average_precision (Notebook 05 PR-AUC definition)",
    "training_scope": "model_development excluding 04B intentional_holdout households",
    "calibration_scope": "calibration and threshold_selection blocks excluding 04B intentional_holdout households",
    "outer_test_labels_loaded": False, "outer_test_scored": False,
    "frozen_input_sha256": source_hashes,
    "software_versions": {"python": platform.python_version(), "pandas": pd.__version__,
        "numpy": np.__version__, "scikit_learn": sklearn.__version__,
        "xgboost": xgboost.__version__, "catboost": catboost.__version__},
    "selected_development_candidates": {target: {k: selected_candidates[target][k] for k in
        ["model_family", "sampling_strategy", "mean_cv_pr_auc", "std_cv_pr_auc", "run_count"]} for target in TARGETS},
    "outputs": output_files,
    "validation_checks": final_audit.to_dict(orient="records"),
}
manifest_path = MANIFEST_DIR / "13_experiment_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")
display(final_audit)
print("All final audit checks passed. Manifest:", manifest_path.relative_to(ROOT))
print(f"Experiment outputs created: {len(output_files) + 1} files under {OUTPUT_DIR.relative_to(ROOT)}")

## Interpretation limits

- The results are development-fold evidence, not a new outer-test evaluation. The original outer test was already used by the completed research and is not reopened here.
- PR-AUC measures ranking on realistic validation prevalence. The 0.5-threshold precision/recall/F1 are diagnostics only.
- Platt calibration and threshold selection use separate chronological development blocks, but the threshold-selection metrics are still development results because that block chooses the threshold.
- If undersampling appears useful, an independent later-time confirmation dataset is needed before claiming improved final generalization or replacing the dissertation's locked model.
- Generated CSVs, plots, bundles, and manifests live only under this experiment's ignored `outputs/` directory. The notebook source remains separate from the existing 18 notebooks.